# 02 - Spatial Baseline + Hybrid (CV-Injected)


In [3]:
import sys
from pathlib import Path

SRC_DIR = (Path.cwd().resolve() / '..' / 'src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Hard-reload local package modules so notebook always sees latest edits.
for name in list(sys.modules.keys()):
    if name == 'course_project' or name.startswith('course_project.'):
        del sys.modules[name]

from course_project.config import ExperimentConfig
from course_project.runner import run_experiment



In [ ]:
# Train spatial baseline + hybrid for side-by-side comparison
import numpy as np
import torch

common_cfg = dict(
    train_dataset='../data/2340_dePablo_networks_OOL_undirected_train5.pt',
    val_dataset='../data/2340_dePablo_networks_OOL_undirected_val200.pt',
    output_root='../results',
    device='cuda',
    pos_dim=2,
    history=1,
    node_features='positions',
    limit=20,
    hidden_size=64,
    n_layers=2,
    learning_rate=5e-5,
    learning_rate_decay=0.997,
    epochs=500,
    val_every=20,
    rollout_steps=100,
    rollout_every=20,
    cv_eval_every=20,
    cv_pratio_target='box',
    train_rollout_steps=2,
    train_rollout_loss_decay=0.9,
)

cv_run_dir = Path('../results/cv_transformer')
cv_stats = torch.load(cv_run_dir / 'train_stats.pt', map_location='cpu', weights_only=False)
cv_epochs = np.asarray(cv_stats['epoch'], dtype=int)
cv_fit_r2 = np.asarray(cv_stats['cv_fit_r2'], dtype=float)
best_idx = int(np.nanargmax(cv_fit_r2))
best_cv_epoch = int(cv_epochs[best_idx])
best_cv_ckpt = cv_run_dir / 'rollout_checkpoints' / f'epoch_{best_cv_epoch:04d}.pt'
print('Using CV checkpoint:', best_cv_ckpt)
print('CV fit R2:', float(cv_fit_r2[best_idx]), 'epoch:', best_cv_epoch)

cfg_spatial = ExperimentConfig(
    run_name='spatial_baseline',
    model_type='spatial',
    model_extras={
        'num_mlp': 3,
    },
    **common_cfg,
)

cfg_hybrid = ExperimentConfig(
    run_name='hybrid',
    model_type='hybrid',
    model_extras={
        'num_mlp': 3,
        'K1': 187,
        'K2': 2,
        'transformer_layers': 1,
        'transformer_heads': 1,
        'transformer_dropout': 0.0,
        'k2_hidden_size': 1,
        'cv_checkpoint_path': str(best_cv_ckpt),
        'cv_inject_scale_init': 3.0,
        'cv_edge_aggr': 'mean',
        'cv_use_local_skip': False,
    },
    **common_cfg,
)

m_hybrid = run_experiment(cfg_hybrid)
m_spatial = run_experiment(cfg_spatial)

hybrid_ckpt = torch.load(Path(common_cfg['output_root']) / 'hybrid' / 'final_checkpoint.pt', map_location='cpu', weights_only=False)
m_hybrid['cv_inject_scale'] = float(hybrid_ckpt['model_state_dict']['cv_inject_scale'])
m_spatial['cv_inject_scale'] = np.nan

import pandas as pd
summary = pd.DataFrame([m_spatial, m_hybrid])[[
    'run_name',
    'model_type',
    'rollout_r2',
    'rollout_pearson_r',
    'rollout_pos_mse',
    'best_rollout_epoch',
    'best_rollout_r2',
    'cv_abs_pearson_r',
    'cv_fit_r2',
    'cv_used',
    'cv_inject_scale',
]]

summary


Using CV checkpoint: ../results/cv_transformer/rollout_checkpoints/epoch_0030.pt
CV fit R2: 0.8726009327280028 epoch: 30
[run] hybrid model=hybrid device=cuda output=../results/hybrid
[run] training...
[train] autoregressive loss steps=2 decay=0.9
